In [1]:
# Imports & DB connection
import pandas as pd
from sqlalchemy import text
from data_collection import engine


In [2]:
# Check 1: How many games do we have per season?
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT season, COUNT(*) as games
        FROM witt_game_logs
        GROUP BY season
        ORDER BY season
    """), conn)
print(df)

   season  games
0    2022    150
1    2023    158
2    2024    161
3    2025    157


In [3]:
# Check 2: Do we have a pitcher entry for every Witt game?
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT 
            COUNT(*) as total_witt_games,
            COUNT(p.game_id) as games_with_pitcher
        FROM witt_game_logs w
        LEFT JOIN pitcher_game_logs p ON w.game_id = p.game_id
    """), conn)
print(df)

   total_witt_games  games_with_pitcher
0               626                 626


In [4]:
# Check 3: Are any pitcher stats wildly out of range?
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT 
            MIN(era) as min_era, MAX(era) as max_era, AVG(era) as avg_era,
            MIN(whip) as min_whip, MAX(whip) as max_whip,
            MIN(k_per_9) as min_k9, MAX(k_per_9) as max_k9,
            COUNT(*) FILTER (WHERE throws IS NULL) as missing_handedness
        FROM pitcher_game_logs
    """), conn)
print(df)

   min_era  max_era   avg_era  min_whip  max_whip  min_k9  max_k9  \
0      0.0    23.62  4.201214      0.17      3.55     0.0   16.76   

   missing_handedness  
0                   0  


In [5]:
# Check 4: Does every opponent_id in witt_game_logs have a park factor?
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT DISTINCT w.opponent_id, w.opponent,
            p.park_factor
        FROM witt_game_logs w
        LEFT JOIN park_factors p ON w.opponent_id = p.team_id
        WHERE p.park_factor IS NULL
    """), conn)
print(f"Opponents missing park factors: {len(df)}")
print(df)

Opponents missing park factors: 0
Empty DataFrame
Columns: [opponent_id, opponent, park_factor]
Index: []


In [6]:
# Check 5: TB distribution — does it look reasonable?
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT tb, COUNT(*) as games
        FROM witt_game_logs
        GROUP BY tb
        ORDER BY tb
    """), conn)
print(df)

    tb  games
0    0    159
1    1    150
2    2    125
3    3     67
4    4     52
5    5     36
6    6     18
7    7      6
8    8      5
9    9      4
10  10      3
11  11      1
